# Примеры кадров по меткам

Показывает по несколько картинок на каждую метку — глазами проверить, что
разметка означает то, что мы думаем.

## Как это работает

Сырьё лежит в бакете архивами: Evocargo одним zip на 7.4 ГБ, CADC двумя tar
по полтора. Качать их ради двадцати картинок незачем, поэтому файлы достаются
поштучно **range-запросами прямо из архива в S3**.

Для zip это дёшево: у него есть оглавление в хвосте, читаем его и сразу берём
нужный член. У tar оглавления нет, приходится идти по заголовкам — но данные
между ними пропускаются перемоткой, а не чтением, так что качается несколько
десятков килобайт вместо полутора гигабайт.

Секреты — как в `eda_manifests.ipynb`: переменные окружения, на удалённом ядре
DataSphere они заводятся секретами проекта.

## 1. Окружение и бакет

In [ ]:
%pip install -q boto3 pyarrow pandas matplotlib pillow

import io, os, socket, sys, tarfile, zipfile
import boto3, pandas as pd, pyarrow.parquet as pq

BUCKET   = os.environ.get("DATASETS_BUCKET", "occlusionnet-clearml-b052c3-datasets")
ENDPOINT = "https://storage.yandexcloud.net"
REGION   = "ru-central1"

missing = [k for k in ("S3_KEY", "S3_SECRET") if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        f"нет ключей {missing}. На удалённом ядре — секреты проекта DataSphere "
        "и перезапуск ядра; локально — .env или экспорт в оболочке.")

s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name=REGION,
                  aws_access_key_id=os.environ["S3_KEY"],
                  aws_secret_access_key=os.environ["S3_SECRET"])

objs = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET):
    objs.extend(page.get("Contents", []))
print("хост:", socket.gethostname(), "| объектов в бакете:", len(objs))

## 2. Манифесты

In [ ]:
frames = []
for o in objs:
    k = o["Key"]
    if k.startswith("manifest/") and k.endswith(".parquet"):
        body = s3.get_object(Bucket=BUCKET, Key=k)["Body"].read()
        frames.append(pq.read_table(io.BytesIO(body)).to_pandas())

df = pd.concat(frames, ignore_index=True)
df = df[df.error.isna()].copy()
print(f"кадров: {len(df):,}")
print("метки:", sorted({l for ls in df.labels for l in ls}))

## 3. Чтение куска файла из S3

`zipfile` и `tarfile` работают с любым объектом, у которого есть `read`, `seek`
и `tell`. Подсовываем им обёртку, делающую range-запрос на каждое чтение.

Дальше два разных подхода, и разница принципиальная.

**Zip** несёт оглавление в хвосте: библиотека читает его и сразу прыгает к
нужному члену. Поверх ставим `BufferedReader` — на замерах это 4 запроса и
около 4% объёма архива.

**Tar** оглавления не имеет. Наивный проход с буферизацией скачивает архив до
нужного места: на пробе вышло 73% файла. Поэтому здесь буферизации нет — тогда
`seek` ничего не читает, — и один раз строится индекс «имя → смещение и
размер». Проход по заголовкам обошёлся в 0.2% объёма, а каждый последующий
кадр достаётся ровно одним запросом. Индекс кешируется на время сессии.

In [ ]:
class S3File(io.RawIOBase):
    """Объект в бакете как обычный seekable-файл поверх range-запросов."""

    def __init__(self, client, bucket, key):
        self.c, self.b, self.k = client, bucket, key
        self.size = client.head_object(Bucket=bucket, Key=key)["ContentLength"]
        self.pos = 0

    def readable(self):  return True
    def seekable(self):  return True
    def tell(self):      return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if   whence == io.SEEK_SET: self.pos = offset
        elif whence == io.SEEK_CUR: self.pos += offset
        else:                       self.pos = self.size + offset
        self.pos = max(0, min(self.pos, self.size))
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        r = self.c.get_object(Bucket=self.b, Key=self.k,
                              Range=f"bytes={self.pos}-{self.pos + n - 1}")
        data = r["Body"].read()
        self.pos += len(data)
        return data

    def readinto(self, buf):
        data = self.read(len(buf))
        buf[:len(data)] = data
        return len(data)


_TAR_INDEX = {}

def tar_index(key):
    """Имя члена -> (смещение данных, размер). Один проход, потом из кеша."""
    if key in _TAR_INDEX:
        return _TAR_INDEX[key]
    raw = S3File(s3, BUCKET, key)          # без BufferedReader: seek не читает
    idx = {}
    with tarfile.open(fileobj=raw, mode="r:") as t:
        for m in t:
            if m.isfile():
                idx[m.name] = (m.offset_data, m.size)
    _TAR_INDEX[key] = idx
    print(f"индекс {key}: {len(idx):,} членов")
    return idx

## 4. Где какой кадр лежит

`raw_path` в манифесте — это путь внутри источника, и он совпадает с именем
члена архива: Evocargo паковался как есть, CADC — командой `tar -C raw/cadc`.
Поэтому отображение однозначное.

In [ ]:
def archive_for(row):
    """Ключ архива в бакете и имя члена внутри него."""
    if row.source == "evocargo_raindrops":
        return "raw/evocargo_raindrops/RaindropsOnWindshield.zip", row.raw_path
    if row.source == "cadc":
        group = row.raw_path.split("/")[0]          # lens_snow | clear_lens
        return f"raw/cadc/{group}.tar", row.raw_path
    raise KeyError(f"неизвестный источник: {row.source}")


def read_member(key, member):
    """Один файл из архива в бакете, без скачивания архива целиком."""
    if key.endswith(".zip"):
        fh = io.BufferedReader(S3File(s3, BUCKET, key), buffer_size=1 << 20)
        try:
            with zipfile.ZipFile(fh) as z:
                try:
                    return z.read(member)
                except KeyError:                 # на случай иной раскладки архива
                    hit = next((n for n in z.namelist() if n.endswith(member)), None)
                    if hit is None:
                        raise
                    return z.read(hit)
        finally:
            fh.close()

    idx = tar_index(key)
    if member not in idx:
        hit = next((n for n in idx if n.endswith(member)), None)
        if hit is None:
            raise KeyError(member)
        member = hit
    offset, size = idx[member]
    raw = S3File(s3, BUCKET, key)
    raw.seek(offset)
    return raw.read(size)

## 5. Выбор примеров

По одному кадру из разных сиквенсов. Иначе получим четыре соседних кадра
одной съёмки: при 10–20 Гц они почти неотличимы, и смотреть на них
бессмысленно.

In [ ]:
SEED = 7
PER_LABEL = 4

def pick(label, n=PER_LABEL):
    if label == "clean":
        sub = df[df.labels.apply(len) == 0]
    else:
        sub = df[df.labels.apply(lambda ls: label in ls)]
    if sub.empty:
        return sub
    # сначала по одному кадру на сиквенс, потом добираем, если сиквенсов мало
    one = (sub.groupby("sequence_id", group_keys=False)
              .apply(lambda g: g.sample(1, random_state=SEED)))
    if len(one) >= n:
        return one.sample(n, random_state=SEED)
    extra = sub.drop(one.index).sample(min(n - len(one), len(sub) - len(one)),
                                       random_state=SEED)
    return pd.concat([one, extra])

LABELS = sorted({l for ls in df.labels for l in ls}) + ["clean"]
for lb in LABELS:
    p = pick(lb)
    print(f"{lb:<12} выбрано {len(p)} кадров из {p.sequence_id.nunique()} сиквенсов")

## 6. Картинки

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

INK, MUTED = "#0b0b0b", "#898781"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "text.color": INK,
                     "figure.dpi": 110})

def show(label, n=PER_LABEL, max_side=520):
    rows = pick(label, n)
    if rows.empty:
        print(f"{label}: примеров нет")
        return
    fig, axes = plt.subplots(1, len(rows), figsize=(3.4 * len(rows), 3.4))
    axes = [axes] if len(rows) == 1 else list(axes)
    for ax, (_, r) in zip(axes, rows.iterrows()):
        key, member = archive_for(r)
        data = read_member(key, member)
        img = Image.open(io.BytesIO(data))
        img.thumbnail((max_side, max_side))         # уменьшаем только для показа
        ax.imshow(img)
        ax.set_axis_off()
        ax.set_title(f"{r.source}\n{r.sequence_id}", fontsize=8, color=MUTED)
    fig.suptitle(label, fontsize=13, fontweight="bold", color=INK, y=1.02)
    fig.tight_layout()
    plt.show()
    print(f"{label}: показано {len(rows)} кадров")

for lb in LABELS:
    show(lb)

## 7. Маски капель

У Evocargo рядом с каждым кадром лежит маска той же формы — в архиве это
`masks/` вместо `images/`. Полезно посмотреть, что именно размечено: наша
метка `raindrops` выведена из непустоты этой маски, и стоит убедиться, что
она действительно про капли.

In [ ]:
import numpy as np

def show_masks(n=3, max_side=520):
    rows = pick("raindrops", n * 3)
    rows = rows[rows.source == "evocargo_raindrops"].head(n)
    if rows.empty:
        print("нет подходящих кадров")
        return
    fig, axes = plt.subplots(1, len(rows), figsize=(3.6 * len(rows), 3.6))
    axes = [axes] if len(rows) == 1 else list(axes)
    for ax, (_, r) in zip(axes, rows.iterrows()):
        key, member = archive_for(r)
        img_b = read_member(key, member)
        msk_b = read_member(key, member.replace("images/", "masks/", 1))

        img = Image.open(io.BytesIO(img_b)).convert("RGB")
        msk = Image.open(io.BytesIO(msk_b)).convert("L").resize(img.size)
        img.thumbnail((max_side, max_side))
        msk = msk.resize(img.size)

        ax.imshow(img)
        # маска поверх полупрозрачным пятном, а не заливкой: под ней должно
        # быть видно, что именно размечено
        ax.imshow(np.ma.masked_where(np.array(msk) == 0, np.array(msk)),
                  cmap="autumn", alpha=0.45)
        ax.set_axis_off()
        ax.set_title(f"{r.sequence_id}  доля маски "
                     f"{(np.array(msk) > 0).mean()*100:.1f}%",
                     fontsize=8, color=MUTED)
    fig.suptitle("raindrops: кадр и маска разметки", fontsize=13,
                 fontweight="bold", color=INK, y=1.02)
    fig.tight_layout()
    plt.show()

show_masks()